In [1]:
import os
import numpy as np
from PIL import Image
from pytorch_fid import fid_score
from tqdm import tqdm
import pandas as pd

from qugen.main.generator.continuous_qgan_model_handler import ContinuousQGANModelHandler

/opt/anaconda3/envs/qugen_env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
image_folder_root_path = 'images_FID/'
n_FID_samples = 10_000

### Disclaimer:
**Make sure to run this notebook after having run the plot generation notebook (i.e., plots_paper.ipynb, patchQGAN_benchmark.ipynb) to ensure that all required samples are available.**

pytorch_fid is used for FID computation. Even though FID was integrated in the qugen package for this project, we use the original pytorch_fid package here for optimal comparability with other works and to speed up the evaluation process as it stores precomputed statistics (especially desirable for the real datasets).

## Precompute 10,000 samples and their statistics for FID evaluation:

Note that this step only needs to be done once per experiment. However, the initial computation may take a while!

In [3]:
experiment_image_stats = {

    # Real datasets:
    'real_MNIST': {
        'path_npy_samples': "apps/logistics/training_data/mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000.npy",
        'epoch': -1,
        'normalized': False,
    },
    'real_MNIST_0_1_2': {
        'path_npy_samples': "apps/logistics/training_data/mnist_0_1_2_32x32_N_18623.npy",
        'epoch': -1,
        'normalized': False,
    },
    'real_MNIST_0_1': {
        'path_npy_samples': "apps/logistics/training_data/mnist_0_1_32x32_N_12665.npy",
        'epoch': -1,
        'normalized': False,
    },
    'real_FashionMNIST': {
        'path_npy_samples': "apps/logistics/training_data/fashion_mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000.npy",
        'epoch': -1,
        'normalized': False,
        'inv_colors': True,
    },
    'real_FashionMNIST_0_1': {
        'path_npy_samples': "apps/logistics/training_data/fashion_mnist_0_1_32x32_N_12000.npy",
        'epoch': -1,
        'normalized': False,
        'inv_colors': True,
    },
    'real_SVHN': {
        'path_npy_samples': "apps/logistics/training_data/svhn_0_32x32_N_45550.npy",
        'epoch': -1,
        'normalized': False,
        'n_channels': 3,
    },

    # Generated data:
    'fake_MNIST_largest': {
        'path_npy_samples': "experiments/continuous_mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000_minmax_qgan_8921f0fc/images_by_mode/epoch=37000_n_per_mode=500_seed=2.npz",  # run plots_paper.ipynb first to ensure sample generation
        # 'epoch': 37000,
    },
    'fake_FashionMNIST_largest': {
        'path_npy_samples': "experiments/continuous_fashion_mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000_minmax_qgan_c5f811fc/images_by_mode/epoch=47000_n_per_mode=500_seed=2.npz",  # run plots_paper.ipynb first to ensure sample generation
        #'epoch': 47000,
        'inv_colors': True,
    },
    'fake_SVHN_largest': {
        'experiment_path': "experiments/continuous_svhn_0_32x32_N_45550_minmax_qgan_ba56d178",
        'epoch': 59100,
        'n_channels': 3,
    },
    'fake_FashionMNIST_no_over': {
        'experiment_path': "experiments/continuous_fashion_mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000_minmax_qgan_0533b651",
        'epoch': 20000,
        'inv_colors': True,
    },
    'fake_FashionMNIST_mid_over': {
        'path_npy_samples': "experiments/continuous_fashion_mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000_minmax_qgan_e098dc80/images_by_mode/epoch=20000_n_per_mode=1000_seed=42.npz",  # run plots_paper.ipynb first to ensure sample generation
        # 'epoch': 20000,
        'inv_colors': True,
    },
    'fake_FashionMNIST_over': {
        'path_npy_samples': "experiments/continuous_fashion_mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000_minmax_qgan_cbf8da2e/images_by_mode/epoch=36800_n_per_mode=1000_seed=42.npz",  # run plots_paper.ipynb first to ensure sample generation
        # 'epoch': 36800,
        'inv_colors': True,
    },
    'fake_FashionMNIST_over_0_1': {
        'path_npy_samples': "experiments/continuous_fashion_mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000_minmax_qgan_cbf8da2e/images_by_mode/epoch=36800_n_per_mode=1250_seed=42.npz",  # run plots_paper.ipynb first to ensure sample generation
        # 'epoch': 36800,
        'inv_colors': True,
        'select_modes': [8, 14, 21, 25,  # t-shirts
                         19, 24, 30, 38  # trousers
                         ],  # only classes 0 (t-shirts) and 1
    },
    'fake_MNIST_0_1_2_abl_agnostic': {
        'experiment_path': "experiments/continuous_mnist_0_1_2_32x32_N_18623_minmax_qgan_7f68e3ee",
        'epoch': 15000,
    },
    'fake_MNIST_0_1_2_abl_specific': {
        'experiment_path': "experiments/continuous_mnist_0_1_2_32x32_N_18623_minmax_qgan_a3edbcf3",
        'epoch': 15000,
    },
    'fake_MNIST_0_1_2_abl_mix_se_circ': {
        'experiment_path': "experiments/continuous_mnist_0_1_2_32x32_N_18623_minmax_qgan_174c8349",
        'epoch': 15000,
    },
    'fake_MNIST_0_1_2_abl_mix_frqi_circ': {
        'experiment_path': "experiments/continuous_mnist_0_1_2_32x32_N_18623_minmax_qgan_7e20b476",
        'epoch': 15000,
    },
    'fake_MNIST_0_1_2_single': {
        'experiment_path': "experiments/continuous_mnist_0_1_2_32x32_N_18623_minmax_qgan_d241d4a5",
        'epoch': 15000,
    },
    'fake_MNIST_0_1_2_multi_no_tuning': {
        'experiment_path': "experiments/continuous_mnist_0_1_2_32x32_N_18623_minmax_qgan_daa843cf",
        'epoch': 10000,
    },
    'fake_MNIST_0_1_2_multi': {
        'experiment_path': "experiments/continuous_mnist_0_1_2_32x32_N_18623_minmax_qgan_a3edbcf3",  # same as abl_specific
        'epoch': 15000,
    },

     "fake_MNIST_depth_8_cls_2": {
        'experiment_path': "experiments/continuous_mnist_0_1_32x32_N_12665_minmax_qgan_f2e2366e",
        'epoch': 29500,
    },

    "fake_MNIST_depth_8": {
        'experiment_path': "experiments/continuous_mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000_minmax_qgan_f3be62bc",
        'epoch': 35500,
    },

    'fake_MNIST_depth_16': {
        'experiment_path': "experiments/continuous_mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000_minmax_qgan_dbe9b988",
        'epoch': 30600,
    },

    'fake_MNIST_depth_32': {
        'experiment_path': "experiments/continuous_mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000_minmax_qgan_c90b8717",
        'epoch': 38500,
    },

    # Generated images from PatchQGAN (Tsang et al. 2023):
    # See patchQGAN_benchmark.ipynb for image generation instructions
    'fake_MNIST_PatchQGAN': None,
    'fake_FashionMNIST_PatchQGAN': None,
}

In [4]:
def reload_qgan_and_sample(experiment_path, epoch, save_path, n_channels=1):
    model_handler = ContinuousQGANModelHandler()
    model_name = os.path.basename(experiment_path)
    print(f"Loading images for {model_name=} iteration {epoch}...")

    model_handler.reload(model_name=model_name, epoch=epoch)
    batch_size = 10
    all_samples = np.empty((n_FID_samples, 32, 32, n_channels)).squeeze()

    for i in tqdm(range(0, n_FID_samples, batch_size)):
        batch_samples = model_handler.sample(batch_size)
        all_samples[i:i+batch_size] = batch_samples.reshape(batch_size, 32, 32, n_channels).squeeze()

    np.save(save_path, all_samples)

def prepare_samples(image_folder, experiment_path=None, epoch=None, path_npy_samples=None, inv_colors=False, n_channels=1, normalized=True, **kwargs):
    """ Samples model if needed and stores them as uncompressed PNG files """
    if path_npy_samples is None:
        path_npy_samples = os.path.join(experiment_path, f"{n_FID_samples}_samples_iteration={epoch}.npy")  # default npy samples path

    # 1) Sample images:
    if not os.path.exists(path_npy_samples):
        if experiment_path is None:
            raise ValueError(f"Provided npy samples path {path_npy_samples} does not exist, and no experiment_path given to generate sample from. Probably, you need to run the plot generation notebook first to generate samples, or make sure that the real datasets are prepared and stored in apps/logistics/training_data/ (see README for more info)?")

        reload_qgan_and_sample(experiment_path, epoch, path_npy_samples, n_channels=n_channels)
    else:
        print(f"{image_folder}: Using existing samples from {path_npy_samples}")

    # Load samples
    images = np.load(path_npy_samples)
    if path_npy_samples.endswith(".npz"):  # stored by mode
        select_modes = kwargs.get('select_modes', None)
        if select_modes is not None:
            images = {k: v for k, v in images.items() if k in select_modes or int(k) in select_modes}
        images = np.array(list(images.values()))
        images = images.reshape(-1, *images.shape[2:]).squeeze()

    # shuffle:
    np.random.seed(42)
    np.random.shuffle(images)

    # Prepare samples:
    images = images[:n_FID_samples]
    n_pixels = images.size // (len(images) * n_channels)
    img_size = round(np.sqrt(n_pixels))
    images = images.reshape(len(images), img_size, img_size, n_channels).squeeze()
    if normalized:  # assuming images are in [0, 1]
        images = 255 * images
    if inv_colors:
        images = 255 - images
    images = images.clip(0, 255).astype(np.uint8)

    # 2) Save images as uncompressed PNGs in image_folder:
    image_folder_path = os.path.join(image_folder_root_path, image_folder)
    try:
        os.makedirs(image_folder_path, exist_ok=False)
        for i, img_array in enumerate(images):
            # 'L' for 8-bit grayscale, 'RGB' for color images
            img = Image.fromarray((img_array).astype(np.uint8),
                                  mode=({1: 'L', 3: 'RGB'}[n_channels]))
            img.save(os.path.join(image_folder_path, f"img_{i}.png"), compress_level=0)
    except FileExistsError:
        print(f"{image_folder_path}: Image folder already exists, skipping.")


In [5]:
def compute_FID_stats(image_folder):

    stats_path = os.path.join(image_folder_root_path, f"{image_folder}.npz")
    if os.path.exists(stats_path):
        print(f"{image_folder}: Skip FID stats, already exist at {stats_path}.")
        return
    image_folder_path = os.path.join(image_folder_root_path, image_folder)

    # Analogous CLI: !python -m pytorch_fid --save-stats {image folder} {stats file}
    fid_score.save_fid_stats(paths=[image_folder_path, stats_path],
                             batch_size=50, device=None, dims=2048,  # default args
                             num_workers=os.cpu_count())

In [6]:
for experiment_key, info in experiment_image_stats.items():
    if info is None:
        print(f"{experiment_key}: No sampling info provided. Assumes that images exist already.")
    else:
        prepare_samples(experiment_key, **info)

    compute_FID_stats(experiment_key)

real_MNIST: Using existing samples from apps/logistics/training_data/mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000.npy
images_FID/real_MNIST: Image folder already exists, skipping.
real_MNIST: Skip FID stats, already exist at images_FID/real_MNIST.npz.
real_MNIST_0_1_2: Using existing samples from apps/logistics/training_data/mnist_0_1_2_32x32_N_18623.npy
images_FID/real_MNIST_0_1_2: Image folder already exists, skipping.
real_MNIST_0_1_2: Skip FID stats, already exist at images_FID/real_MNIST_0_1_2.npz.
real_MNIST_0_1: Using existing samples from apps/logistics/training_data/mnist_0_1_32x32_N_12665.npy
images_FID/real_MNIST_0_1: Image folder already exists, skipping.
real_MNIST_0_1: Skip FID stats, already exist at images_FID/real_MNIST_0_1.npz.
real_FashionMNIST: Using existing samples from apps/logistics/training_data/fashion_mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000.npy
images_FID/real_FashionMNIST: Image folder already exists, skipping.
real_FashionMNIST: Skip FID stats, already exist at 

## Evaluate FID scores for selected experiments:
Define pairs of (generated images stats, real images stats) for FID evaluation.

In [7]:
fid_eval_pairs = {

    'MNIST_large': ("fake_MNIST_largest", "real_MNIST"),
    'FashionMNIST_large': ("fake_FashionMNIST_largest", "real_FashionMNIST"),
    'SVHN_large': ("fake_SVHN_largest", "real_SVHN"),

    'FashionMNIST_no_overmoding': ("fake_FashionMNIST_no_over", "real_FashionMNIST"),
    'FashionMNIST_mid_overmoding': ( "fake_FashionMNIST_mid_over", "real_FashionMNIST"),
    'FashionMNIST_overmoding': ("fake_FashionMNIST_over", "real_FashionMNIST"),

    'FashionMNIST_overmoding_0_1': ("fake_FashionMNIST_over_0_1", "real_FashionMNIST_0_1"),

    'MNIST_0_1_2_abl_agnostic': ("fake_MNIST_0_1_2_abl_agnostic", "real_MNIST_0_1_2"),  # Sub-fig a)
    'MNIST_0_1_2_abl_mix_se_circ': ("fake_MNIST_0_1_2_abl_mix_se_circ", "real_MNIST_0_1_2"),  # Sub-fig b)
    'MNIST_0_1_2_abl_mix_frqi_circ': ("fake_MNIST_0_1_2_abl_mix_frqi_circ", "real_MNIST_0_1_2"),    # Sub-fig c)
    'MNIST_0_1_2_abl_specific': ("fake_MNIST_0_1_2_abl_specific", "real_MNIST_0_1_2"),    # Sub-fig d)

    'MNIST_0_1_2_single_mode': ("fake_MNIST_0_1_2_single", "real_MNIST_0_1_2"),
    'MNIST_0_1_2_multi_no_tuning': ("fake_MNIST_0_1_2_multi_no_tuning", "real_MNIST_0_1_2"),
    'MNIST_0_1_2_multi_tuning': ("fake_MNIST_0_1_2_multi", "real_MNIST_0_1_2"),

    'MNIST_depth_8_cls_2': ("fake_MNIST_depth_8_cls_2", "real_MNIST_0_1"),
    'MNIST_depth_8': ("fake_MNIST_depth_8", "real_MNIST"),
    'MNIST_depth_16': ("fake_MNIST_depth_16", "real_MNIST"),
    'MNIST_depth_32': ("fake_MNIST_depth_32", "real_MNIST"),

    # Results from PatchQGAN (Tsang et al. 2023):

    'PatchQGAN_MNIST': ("fake_MNIST_PatchQGAN", "real_MNIST_0_1_2"),
    'PatchQGAN_FashionMNIST': ("fake_FashionMNIST_PatchQGAN", "real_FashionMNIST_0_1"),
}


In [8]:

fid_evals = {}


for experiment_key, (gen_stats_name, real_stats_name) in fid_eval_pairs.items():
    gen_stats = os.path.join(image_folder_root_path, gen_stats_name)
    # ensure .npz extension
    if not gen_stats.endswith('.npz'):
        gen_stats += '.npz'
    real_stats = os.path.join(image_folder_root_path, real_stats_name)
    if not real_stats.endswith('.npz'):
        real_stats += '.npz'

    # Analogous CLI: !python -m pytorch_fid {gen_stats} {real_stats}
    fid_value = fid_score.calculate_fid_given_paths(paths=[gen_stats, real_stats],
                                                    batch_size=50, device=None, dims=2048)  # default args
    print(f"{experiment_key} FID: {fid_value}")
    fid_evals[experiment_key] = fid_value

MNIST_large FID: 118.26599124555793
FashionMNIST_large FID: 90.86686902210946
SVHN_large FID: 84.15531464010411
FashionMNIST_no_overmoding FID: 103.39428318399712
FashionMNIST_mid_overmoding FID: 97.25756361798722
FashionMNIST_overmoding FID: 70.03164257747832
FashionMNIST_overmoding_0_1 FID: 60.29539355358955
MNIST_0_1_2_abl_agnostic FID: 279.24027915471993
MNIST_0_1_2_abl_mix_se_circ FID: 341.0890463211713
MNIST_0_1_2_abl_mix_frqi_circ FID: 203.61815996125682
MNIST_0_1_2_abl_specific FID: 152.2199934648881
MNIST_0_1_2_single_mode FID: 216.5084470105018
MNIST_0_1_2_multi_no_tuning FID: 200.91518309299246
MNIST_0_1_2_multi_tuning FID: 152.2199934648881
MNIST_depth_8_cls_2 FID: 108.02004200144651
MNIST_depth_8 FID: 157.29212847526003
MNIST_depth_16 FID: 138.72073884932254
MNIST_depth_32 FID: 108.9882667823502
PatchQGAN_MNIST FID: 206.5098709184375
PatchQGAN_FashionMNIST FID: 178.85663769540722


# Evaluate MMD scores for same experiments:

In [9]:
%load_ext autoreload
%autoreload 2

In [10]:
from qugen.main.data.mmd import mmd_linear, mmd_rbf, mmd_poly
import glob
import re

In [11]:
mmd_eval_pairs = fid_eval_pairs.copy()

In [19]:
mmd_evals = {}

def load_and_flatten_images(folder_path, img_size=None):
    """
    Loads all PNGs in a folder, flattens them into 1D arrays,
    and normalizes pixel values to [0, 1].
    """
    file_paths = glob.glob(os.path.join(folder_path, "*.png"))
    if not file_paths:
        raise FileNotFoundError(f"No PNG images found in {folder_path}")

    img_list = []
    for fp in file_paths:
        img = Image.open(fp)
        if img_size is not None and img_size ** 2 != img.size[0] * img.size[1]:
            #raise warning if asked for upsampling:
            assert img_size < img.size[0] and img_size < img.size[1], \
                f"Stored image resolution smaller {img.size} than requested image size {(img_size, img_size)}."
            img = img.resize((img_size, img_size), Image.Resampling.LANCZOS)
        # Convert to numpy array, flatten to 1D, and normalize
        # Normalization is crucial for RBF/Poly kernels to prevent numerical overflow
        img_array = np.array(img, dtype=np.float32).flatten() / 255.0
        img_list.append(img_array)

    return np.stack(img_list), img.size[0]

for experiment_key, (gen_name, real_name) in mmd_eval_pairs.items():
    assert real_name.startswith("real"), f"{real_name} does not start with 'real', check your naming convention."

    gen_folder_path = os.path.join(image_folder_root_path, gen_name)
    real_folder_path = os.path.join(image_folder_root_path, "test_" + real_name)

    # Ensure that test dataset is present:
    prep_specs = experiment_image_stats[real_name]
    prep_specs['path_npy_samples'] = os.path.join(os.path.split(os.path.split(prep_specs['path_npy_samples'])[0])[0],
                                                  "test_data",
                                                  re.sub(r"_N_\d+", "", os.path.split(prep_specs['path_npy_samples'])[1]))
    print(prep_specs['path_npy_samples'], real_folder_path)
    #continue
    prepare_samples("test_" + real_name, **prep_specs)

    print(f"Loading images for {experiment_key}...")
    X, img_size = load_and_flatten_images(gen_folder_path)
    Y, _ = load_and_flatten_images(real_folder_path, img_size=img_size)

    print(f"Computing MMD scores (Matrix shapes: X={X.shape}, Y={Y.shape})...")

    # Compute MMD scores
    print("Min-max pixel values per dataset:", np.min(X), np.max(X), np.min(Y), np.max(Y))
    val_linear = mmd_linear(X, Y)
    val_rbf = mmd_rbf(X, Y, gamma=1./(img_size**2))
    val_poly = mmd_poly(X, Y, gamma=1./(img_size**2))

    print(f"{experiment_key} MMD - Linear: {val_linear:.4f} | RBF: {val_rbf:.4f} | Poly: {val_poly:.4f}\n")

    # Store the scores
    mmd_evals[experiment_key] = {
        'linear': val_linear,
        'rbf': val_rbf,
        'poly': val_poly
    }

apps/logistics/test_data/mnist_0_1_2_3_4_5_6_7_8_9_32x32.npy images_FID/test_real_MNIST
test_real_MNIST: Using existing samples from apps/logistics/test_data/mnist_0_1_2_3_4_5_6_7_8_9_32x32.npy
images_FID/test_real_MNIST: Image folder already exists, skipping.
Loading images for MNIST_large...
Computing MMD scores (Matrix shapes: X=(10000, 1024), Y=(10000, 1024))...
Min-max pixel values per dataset: 0.0 0.99607843 0.0 1.0
MNIST_large MMD - Linear: 0.5983 | RBF: 0.0018 | Poly: 0.0001

apps/logistics/test_data/fashion_mnist_0_1_2_3_4_5_6_7_8_9_32x32.npy images_FID/test_real_FashionMNIST
test_real_FashionMNIST: Using existing samples from apps/logistics/test_data/fashion_mnist_0_1_2_3_4_5_6_7_8_9_32x32.npy
images_FID/test_real_FashionMNIST: Image folder already exists, skipping.
Loading images for FashionMNIST_large...
Computing MMD scores (Matrix shapes: X=(10000, 1024), Y=(10000, 1024))...
Min-max pixel values per dataset: 0.0 1.0 0.0 1.0
FashionMNIST_large MMD - Linear: 0.4542 | RBF: 0

In [20]:
extra_info = {

    'MNIST_large': {"fig_label": "fig:large_model:mnist"},
    'FashionMNIST_large': {"fig_label": "fig:large_model:fmnist"},
    'SVHN_large': {"fig_label": "fig:color_images"},

    'FashionMNIST_no_overmoding': {"fig_label": "fig:more_modes:1"},
    'FashionMNIST_mid_overmoding': {"fig_label": "fig:more_modes:2"},
    'FashionMNIST_overmoding': {"fig_label": "fig:more_modes:4"},

    'FashionMNIST_overmoding_0_1': {"fig_label": "fig:patch_qgan:ours_fmnist"},

    'MNIST_0_1_2_abl_agnostic': {"fig_label": "fig:agno_circ_amp_enc"},  # Sub-fig a)
    'MNIST_0_1_2_abl_mix_se_circ': {"fig_label": "fig:agno_circ_frqi_enc"},  # Sub-fig b)
    'MNIST_0_1_2_abl_mix_frqi_circ': {"fig_label": "fig:spef_circ_amp_enc"},    # Sub-fig c)
    'MNIST_0_1_2_abl_specific': {"fig_label": ["fig:spef_circ_frqi_enc", "fig:patch_qgan:ours_mnist"]},    # Sub-fig d)

    'MNIST_0_1_2_single_mode': {"fig_label": "fig:comp_noise:unimodal"},
    'MNIST_0_1_2_multi_no_tuning': {"fig_label": "fig:comp_noise:no_tuning"},
    'MNIST_0_1_2_multi_tuning': {"fig_label": "fig:comp_noise:multimodal"},

    'MNIST_depth_8_cls_2': {"fig_label": "fig:classes_model_size:2c_8l"},
    'MNIST_depth_8': {"fig_label": "fig:classes_model_size:10c_8l"},
    'MNIST_depth_16': {"fig_label": "fig:classes_model_size:10c_16l"},
    'MNIST_depth_32': {"fig_label": "fig:classes_model_size:10c_32l"},

    'PatchQGAN_MNIST': {"fig_label": "fig:patch_qgan:patch_mnist"},
    'PatchQGAN_FashionMNIST': {"fig_label": "fig:patch_qgan:patch_fmnist"},
}

In [31]:
fig_refs_dict = dict()
dataset_dict = dict()
n_classes_dict = dict()
for experiment_key, extra in extra_info.items():
    # fig ref
    fig_label = extra['fig_label']
    if isinstance(fig_label, str):
        fig_label = [fig_label]
    latex_fig_ref = [f"\\ref{{{label}}}" for label in fig_label]
    fig_refs_dict[experiment_key] = latex_fig_ref
    # dataset info (infer from real dataset)
    real_dataset_id = fid_eval_pairs[experiment_key][1]
    assert real_dataset_id.startswith("real"), f"{real_dataset_id} does not start with 'real', check your naming convention."
    dataset_id_split = real_dataset_id.split("_")
    dataset_name = dataset_id_split[1]  # post process dataset name follows
    dataset_name = {'FashionMNIST': "Fashion-MNIST"}.get(dataset_name, dataset_name)
    dataset_dict[experiment_key] = dataset_name

    n_classes = len(dataset_id_split[2:])  # number of classes is inferred from the real dataset nam; post process follows
    n_classes = 10 if n_classes == 0 else n_classes
    n_classes_dict[experiment_key] = n_classes

In [32]:
experiment_key_order = [
    'MNIST_large',
    'FashionMNIST_large',
    'SVHN_large',
    'MNIST_0_1_2_abl_agnostic',
    'MNIST_0_1_2_abl_mix_se_circ',
    'MNIST_0_1_2_abl_mix_frqi_circ',
    'MNIST_0_1_2_abl_specific',
    'MNIST_0_1_2_single_mode',
    'MNIST_0_1_2_multi_no_tuning',
    'MNIST_0_1_2_multi_tuning',
    'FashionMNIST_no_overmoding',
    'FashionMNIST_mid_overmoding',
    'FashionMNIST_overmoding',
    'MNIST_depth_8_cls_2',
    'MNIST_depth_8',
    'MNIST_depth_16',
    'MNIST_depth_32',
    'PatchQGAN_MNIST',
    'PatchQGAN_FashionMNIST',
    'FashionMNIST_overmoding_0_1',
]
assert set(experiment_key_order) == set(mmd_evals.keys()), "Experiment keys in different dicts do not match. Please check your dicts and the experiment_key_order list."

In [33]:
# Use pandas and export as LaTeX table (include FID score, assuming FID evals is a superset):
table = pd.DataFrame(index=experiment_key_order)

# Merge in order to have columns left to right
table = table.join(pd.Series(fig_refs_dict, name="Figure").to_frame(), how='left')

table = table.join(pd.DataFrame.from_dict(dataset_dict, orient='index', columns=["Dataset"]), how='left')
table = table.join(pd.DataFrame.from_dict(n_classes_dict, orient='index', columns=["Num.~classes"]), how='left')

table = table.join(pd.DataFrame.from_dict(fid_evals, orient='index', columns=["fid"]), how='left')

table = table.join(pd.DataFrame.from_dict(mmd_evals, orient='index'), how='left')




table

,Figure,Dataset,Num.~classes,fid,linear,rbf,poly
MNIST_large,[\ref{fig:large_model:mnist}],MNIST,10,118.265991,0.598259,0.001776,0.000089
FashionMNIST_large,[\ref{fig:large_model:fmnist}],Fashion-MNIST,10,90.866869,0.454224,0.000875,0.000666
SVHN_large,[\ref{fig:color_images}],SVHN,10,84.155315,3.998169,0.005388,0.011632
MNIST_0_1_2_abl_agnostic,[\ref{fig:agno_circ_amp_enc}],MNIST,3,279.240279,12.360458,0.024744,0.001330
MNIST_0_1_2_abl_mix_se_circ,[\ref{fig:agno_circ_frqi_enc}],MNIST,3,341.089046,54.535645,0.099208,0.010029
MNIST_0_1_2_abl_mix_frqi_circ,[\ref{fig:spef_circ_amp_enc}],MNIST,3,203.618160,1.550545,0.004415,0.000272
MNIST_0_1_2_abl_specific,"[\ref{fig:spef_circ_frqi_enc}, \ref{fig:patch_...",MNIST,3,152.219993,1.415382,0.003484,0.000195
MNIST_0_1_2_single_mode,[\ref{fig:comp_noise:unimodal}],MNIST,3,216.508447,2.168846,0.005257,0.000306
MNIST_0_1_2_multi_no_tuning,[\ref{fig:comp_noise:no_tuning}],MNIST,3,200.915183,1.639641,0.004703,0.000299
MNIST_0_1_2_multi_tuning,[\ref{fig:comp_noise:multimodal}],MNIST,3,152.219993,1.415382,0.003484,0.000195


In [34]:
# Calculate correlations between MMD (split by kernel) and FID
print("Corr linear", table['fid'].corr(table['linear'], method="pearson"), table['fid'].corr(table['linear'], method="spearman"))
print("Corr rbf", table['fid'].corr(table['rbf'], method="pearson"), table['fid'].corr(table['rbf'], method="spearman"))
print("Corr poly", table['fid'].corr(table['poly'], method="pearson"), table['fid'].corr(table['poly'], method="spearman"))

Corr linear 0.7114510063636347 0.7712565838976674
Corr rbf 0.7389158367070897 0.8209179834461999
Corr poly 0.2580583445849799 -0.0052671181339352885


In [35]:
table = table.explode("Figure")

header_hierarchy = [
    ("", "Figure"),
    ("", "Dataset"),
    ("", "Num.~classes"),
    ("", "FID"),
    ("MMD", "Linear"),
    ("MMD", "Poly"),
    ("MMD", "RBF"),
]
table.columns = pd.MultiIndex.from_tuples(header_hierarchy)

def to_latex_sci(x):
    """Converts a float to proper LaTeX scientific notation."""
    if pd.isna(x):
        return x

    # Format to standard scientific string first (e.g., "1.23e-04")
    sci_str = f"{x:.2e}"
    base, exponent = sci_str.split('e')

    # Cast exponent to int to drop the leading zeros and plus signs (e.g., "-04" -> "-4", "+02" -> "2")
    exponent = int(exponent)

    # Return formatted LaTeX math string
    return f"${base} \\times 10^{{{exponent}}}$"

# Apply the custom LaTeX formatter
table[("MMD", "RBF")] = table[("MMD", "RBF")].apply(to_latex_sci)
table[("MMD", "Poly")] = table[("MMD", "Poly")].apply(to_latex_sci)

# Export to LaTeX
latex_code = table.to_latex(
    escape=False,
    float_format="%.4f",
    index=False,
    index_names=False,
    multicolumn=True,         # Tells pandas to use \multicolumn for the "MMD" header
    multicolumn_format='c',  # Centers the "MMD" text over the three columns
    multirow=True            # Manually put multirow
)

print(latex_code)

\begin{tabular}{llrrrll}
\toprule
\multicolumn{4}{c}{} & \multicolumn{3}{c}{MMD} \\
Figure & Dataset & Num.~classes & FID & Linear & Poly & RBF \\
\midrule
\ref{fig:large_model:mnist} & MNIST & 10 & 118.2660 & 0.5983 & $1.78 \times 10^{-3}$ & $8.93 \times 10^{-5}$ \\
\ref{fig:large_model:fmnist} & Fashion-MNIST & 10 & 90.8669 & 0.4542 & $8.75 \times 10^{-4}$ & $6.66 \times 10^{-4}$ \\
\ref{fig:color_images} & SVHN & 10 & 84.1553 & 3.9982 & $5.39 \times 10^{-3}$ & $1.16 \times 10^{-2}$ \\
\ref{fig:agno_circ_amp_enc} & MNIST & 3 & 279.2403 & 12.3605 & $2.47 \times 10^{-2}$ & $1.33 \times 10^{-3}$ \\
\ref{fig:agno_circ_frqi_enc} & MNIST & 3 & 341.0890 & 54.5356 & $9.92 \times 10^{-2}$ & $1.00 \times 10^{-2}$ \\
\ref{fig:spef_circ_amp_enc} & MNIST & 3 & 203.6182 & 1.5505 & $4.41 \times 10^{-3}$ & $2.72 \times 10^{-4}$ \\
\ref{fig:spef_circ_frqi_enc} & MNIST & 3 & 152.2200 & 1.4154 & $3.48 \times 10^{-3}$ & $1.95 \times 10^{-4}$ \\
\ref{fig:patch_qgan:ours_mnist} & MNIST & 3 & 152.2200 & 1.